In [1]:
import os 

In [2]:
import os
import json
import traceback

import pandas as pd
import PyPDF2

from dotenv import load_dotenv

from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
load_dotenv()

True

In [4]:
api = os.getenv("MISTRAL")

In [6]:
llm = ChatMistralAI(
    api_key=api,
    model="mistral-large-latest",
    temperature=1,
)

In [7]:
llm

ChatMistralAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Mistral Large (latest)', 'release_date': '2024-11-01', 'last_updated': '2025-12-02', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 262144, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': True, 'temperature': True}, client=<httpx.Client object at 0x0000025C1F7F34D0>, async_client=<httpx.AsyncClient object at 0x0000025C1F800050>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model='mistral-large-latest', temperature=1.0, model_kwargs={})

In [64]:
with open("../response.json", "r") as f:
    response = json.load(f)

In [35]:
print(response)

{'quiz': [{'MCQS number ': 'Question 1', 'options': {'A': 'Option A', 'B': 'Option B', 'C': 'Option C', 'D': 'Option D'}, 'correct_answer': 'B'}, {'MCQS number ': 'Question 2', 'options': {'A': 'Option A', 'B': 'Option B', 'C': 'Option C', 'D': 'Option D'}, 'correct_answer': 'C'}]}


In [65]:
template = """
You are an expert MCQ creator with strong knowledge of educational assessment.

Your task is to create a quiz based ONLY on the provided text.

TEXT:
{text}

Instructions:
- Generate exactly {number} multiple-choice questions.
- multiple choice question from {subject} subject.
- The difficulty level should be {tone}.
- Every question must have exactly four options labeled A, B, C, and D..
- Only one option should be correct.
- Do not create questions from information that is not present in the provided text.
- Ensure the questions are clear, relevant, and non-repetitive.
- Make sure the quiz is suitable for students.
- Return ONLY valid JSON.
- Do not include explanations, markdown, or additional text.

The JSON format must exactly match the following schema:

{response}
"""

In [66]:
prompt = PromptTemplate (
    input_variables=["text", "number", "subject", "tone", "response"],
    template=template,
)

In [67]:
chain1 = prompt | llm 

In [68]:
print(chain1)

first=PromptTemplate(input_variables=['number', 'response', 'subject', 'text', 'tone'], input_types={}, partial_variables={}, template='\nYou are an expert MCQ creator with strong knowledge of educational assessment.\n\nYour task is to create a quiz based ONLY on the provided text.\n\nTEXT:\n{text}\n\nInstructions:\n- Generate exactly {number} multiple-choice questions.\n- multiple choice question from {subject} subject.\n- The difficulty level should be {tone}.\n- Every question must have exactly four options labeled A, B, C, and D..\n- Only one option should be correct.\n- Do not create questions from information that is not present in the provided text.\n- Ensure the questions are clear, relevant, and non-repetitive.\n- Make sure the quiz is suitable for students.\n- Return ONLY valid JSON.\n- Do not include explanations, markdown, or additional text.\n\nThe JSON format must exactly match the following schema:\n\n{response}\n') middle=[] last=ChatMistralAI(metadata={'lc_versions': {

In [69]:
template2 = """You are an expert English examiner and MCQ evaluator.

Subject:
{subject}


Task:
You must evaluate the given multiple choice questions based on cognitive and analytical level of students.

Instructions:
- Check if questions are clear, correct, and meaningful
- Check difficulty level according to students' understanding
- Identify any grammar or conceptual mistakes
- If quiz is not appropriate for students' cognitive level, suggest regeneration

Final Output Rules:
1. Provide a complete analysis in exactly 50 words
2. Clearly state whether quiz is GOOD or NEEDS IMPROVEMENT
3. If needed, suggest to regenerate improved version
4. Be strict and professional like an examiner
Quiz MCQs:
{quiz}

"""

In [70]:
prompt2 = PromptTemplate (
    input_variables=["subject","quiz"],
    template=template2,
)

In [71]:
chain2 = prompt2 | llm 

In [72]:
with open("../data.txt", "r") as f:
    text = f.read()

In [73]:
text 
numbers = 10 
subject = "AI"
tone = "medium"
response_json = response


In [74]:
import json
json.dumps(response_json)

'{"quiz": [{"MCQS number and question ": "Question 1", "question": "What is the capital of France?", "options": {"A": "Option A", "B": "Option B", "C": "Option C", "D": "Option D"}, "correct_answer": "B"}, {"MCQS number and question ": "Question 2", "question": "What is the capital of pakistan?", "options": {"A": "Option A", "B": "Option B", "C": "Option C", "D": "Option D"}, "correct_answer": "C"}]}'

In [75]:
quiz = chain1.invoke({
    "text": text,
    "number": numbers,
    "subject": subject,
    "tone": tone,
    "response": json.dumps(response_json)
})

In [76]:
review = chain2.invoke({
    "subject": subject,
    "quiz": quiz
})

In [77]:
print ("quiz: ", review)

quiz:  content='**Analysis (50 words):**\nThe quiz is **GOOD**—clear, conceptually accurate, and well-structured. Questions span foundational to advanced AI knowledge, suitable for intermediate learners. No grammatical errors; options are plausible. Cognitive demand aligns with analytical evaluation. Minor improvement: Question 3’s "mentioned in the text" assumes prior context; rephrase if standalone.\n\n**Verdict:** GOOD. No regeneration needed.' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 1483, 'total_tokens': 1574, 'completion_tokens': 91, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-large-latest', 'model': 'mistral-large-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'} id='lc_run--019f3c23-6d76-7521-be14-7c3fb3511347-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1483, 'output_tokens': 91, 'total_tokens': 1574}


In [78]:
print(quiz)

content='```json\n{\n  "quiz": [\n    {\n      "MCQS number and question": "Question 1",\n      "question": "What is the primary purpose of generative AI models?",\n      "options": {\n        "A": "To store large datasets",\n        "B": "To generate new data based on learned patterns from training data",\n        "C": "To classify existing data into categories",\n        "D": "To delete outdated information"\n      },\n      "correct_answer": "B"\n    },\n    {\n      "MCQS number and question": "Question 2",\n      "question": "Which architecture forms the basis of large language models (LLMs) in generative AI?",\n      "options": {\n        "A": "Recurrent Neural Networks (RNNs)",\n        "B": "Convolutional Neural Networks (CNNs)",\n        "C": "Transformer architecture",\n        "D": "Support Vector Machines (SVMs)"\n      },\n      "correct_answer": "C"\n    },\n    {\n      "MCQS number and question": "Question 3",\n      "question": "Which of the following is NOT a generati

In [79]:
print(quiz.content)

```json
{
  "quiz": [
    {
      "MCQS number and question": "Question 1",
      "question": "What is the primary purpose of generative AI models?",
      "options": {
        "A": "To store large datasets",
        "B": "To generate new data based on learned patterns from training data",
        "C": "To classify existing data into categories",
        "D": "To delete outdated information"
      },
      "correct_answer": "B"
    },
    {
      "MCQS number and question": "Question 2",
      "question": "Which architecture forms the basis of large language models (LLMs) in generative AI?",
      "options": {
        "A": "Recurrent Neural Networks (RNNs)",
        "B": "Convolutional Neural Networks (CNNs)",
        "C": "Transformer architecture",
        "D": "Support Vector Machines (SVMs)"
      },
      "correct_answer": "C"
    },
    {
      "MCQS number and question": "Question 3",
      "question": "Which of the following is NOT a generative AI application mentioned in the t

In [80]:
clean = quiz.content.replace("```json", "").replace("```", "").strip()
quiz_dict = json.loads(clean)

In [81]:
print(quiz_dict)

{'quiz': [{'MCQS number and question': 'Question 1', 'question': 'What is the primary purpose of generative AI models?', 'options': {'A': 'To store large datasets', 'B': 'To generate new data based on learned patterns from training data', 'C': 'To classify existing data into categories', 'D': 'To delete outdated information'}, 'correct_answer': 'B'}, {'MCQS number and question': 'Question 2', 'question': 'Which architecture forms the basis of large language models (LLMs) in generative AI?', 'options': {'A': 'Recurrent Neural Networks (RNNs)', 'B': 'Convolutional Neural Networks (CNNs)', 'C': 'Transformer architecture', 'D': 'Support Vector Machines (SVMs)'}, 'correct_answer': 'C'}, {'MCQS number and question': 'Question 3', 'question': 'Which of the following is NOT a generative AI application mentioned in the text?', 'options': {'A': 'ChatGPT', 'B': 'DALL-E', 'C': 'Tesla Autopilot', 'D': 'Midjourney'}, 'correct_answer': 'C'}, {'MCQS number and question': 'Question 4', 'question': 'In 

In [82]:
print(json.dumps(quiz_dict, indent=4))

{
    "quiz": [
        {
            "MCQS number and question": "Question 1",
            "question": "What is the primary purpose of generative AI models?",
            "options": {
                "A": "To store large datasets",
                "B": "To generate new data based on learned patterns from training data",
                "C": "To classify existing data into categories",
                "D": "To delete outdated information"
            },
            "correct_answer": "B"
        },
        {
            "MCQS number and question": "Question 2",
            "question": "Which architecture forms the basis of large language models (LLMs) in generative AI?",
            "options": {
                "A": "Recurrent Neural Networks (RNNs)",
                "B": "Convolutional Neural Networks (CNNs)",
                "C": "Transformer architecture",
                "D": "Support Vector Machines (SVMs)"
            },
            "correct_answer": "C"
        },
        {
     

In [83]:
quiz_table_data = []

for value in quiz_dict["quiz"]:

    mcq = value["question"]

    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
        ]
    )

    correct = value["correct_answer"]

    quiz_table_data.append(
        {
            "MCQ": mcq,
            "Choices": options,
            "Correct Answer": correct
        }
    )



In [85]:
quiz_table_data

[{'MCQ': 'What is the primary purpose of generative AI models?',
  'Choices': 'A: To store large datasets | B: To generate new data based on learned patterns from training data | C: To classify existing data into categories | D: To delete outdated information',
  'Correct Answer': 'B'},
 {'MCQ': 'Which architecture forms the basis of large language models (LLMs) in generative AI?',
  'Choices': 'A: Recurrent Neural Networks (RNNs) | B: Convolutional Neural Networks (CNNs) | C: Transformer architecture | D: Support Vector Machines (SVMs)',
  'Correct Answer': 'C'},
 {'MCQ': 'Which of the following is NOT a generative AI application mentioned in the text?',
  'Choices': 'A: ChatGPT | B: DALL-E | C: Tesla Autopilot | D: Midjourney',
  'Correct Answer': 'C'},
 {'MCQ': 'In which sector has generative AI NOT been explicitly mentioned as being used?',
  'Choices': 'A: Healthcare | B: Agriculture | C: Entertainment | D: Finance',
  'Correct Answer': 'B'},
 {'MCQ': 'What negative impact of gene

In [90]:
quiz_table = pd.DataFrame(quiz_table_data)

print(quiz_table)

                                                 MCQ  \
0  What is the primary purpose of generative AI m...   
1  Which architecture forms the basis of large la...   
2  Which of the following is NOT a generative AI ...   
3  In which sector has generative AI NOT been exp...   
4  What negative impact of generative AI is highl...   
5  Which concept introduced by Andrey Markov in 1...   
6  What was the name of Harold Cohen's pioneering...   
7  What environmental concern is associated with ...   
8  Which of the following was a use case for gene...   
9  What form of input do generative AI models oft...   

                                             Choices Correct Answer  
0  A: To store large datasets | B: To generate ne...              B  
1  A: Recurrent Neural Networks (RNNs) | B: Convo...              C  
2  A: ChatGPT | B: DALL-E | C: Tesla Autopilot | ...              C  
3  A: Healthcare | B: Agriculture | C: Entertainm...              B  
4  A: Models are trained on copyr

In [94]:
quiz_table.to_csv("../quiz_table.csv", index=False)